In [2]:



# import tkinter as tk
# import threading
# import copy
# import time

# # Constants
# dialog = None
# PIECE_WEIGHTS = {'K': 0, 'Q': 9, 'R': 5, 'B': 3, 'N': 3, 'P': 1}
# FILES = "abcdefgh"
# RANKS = "12345678"

# class Move:
#     def __init__(self, start_sq, end_sq, board, is_en_passant=False, is_castle=False, promotion_choice=None):
#         self.start_row, self.start_col = start_sq
#         self.end_row, self.end_col = end_sq
#         self.piece_moved = board[self.start_row][self.start_col]
#         self.piece_captured = board[self.end_row][self.end_col]
#         self.is_en_passant = is_en_passant
#         self.is_castle = is_castle
#         self.promotion_choice = promotion_choice
#         self.prev_en_passant = None
#         self.prev_castle_rights = None

#     def get_chess_notation(self):
#         return f"{FILES[self.start_col]}{RANKS[7-self.start_row]}{FILES[self.end_col]}{RANKS[7-self.end_row]}" + (
#             f"={self.promotion_choice}" if self.promotion_choice else "")

# class Piece:
#     def __init__(self, color):
#         self.color = color
#     def get_valid_moves(self, row, col, board, en_passant_target, castle_rights):
#         raise NotImplementedError

# class King(Piece):
#     def __init__(self, color):
#         super().__init__(color)
#         self.name = 'K'
#     def get_valid_moves(self, r, c, board, en_passant_target, castle_rights):
#         moves = []
#         directions = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
#         for dr, dc in directions:
#             rr, cc = r+dr, c+dc
#             if 0 <= rr < 8 and 0 <= cc < 8:
#                 target = board[rr][cc]
#                 if target is None or target.color != self.color:
#                     moves.append(Move((r,c), (rr,cc), board))
#         # Castling
#         rights = castle_rights[self.color]
#         if rights['K'] and board[r][c+1] is None and board[r][c+2] is None:
#             moves.append(Move((r,c),(r,c+2), board, is_castle=True))
#         if rights['Q'] and board[r][c-1] is None and board[r][c-2] is None and board[r][c-3] is None:
#             moves.append(Move((r,c),(r,c-2), board, is_castle=True))
#         return moves

# class Queen(Piece):
#     def __init__(self, color):
#         super().__init__(color)
#         self.name = 'Q'
#     def get_valid_moves(self, r, c, board, en_passant_target, castle_rights):
#         return Rook(self.color).get_valid_moves(r,c,board,en_passant_target,castle_rights) + \
#                Bishop(self.color).get_valid_moves(r,c,board,en_passant_target,castle_rights)

# class Rook(Piece):
#     def __init__(self, color):
#         super().__init__(color)
#         self.name = 'R'
#     def get_valid_moves(self, r, c, board, en_passant_target, castle_rights):
#         moves = []
#         dirs = [(-1,0),(1,0),(0,-1),(0,1)]
#         for dr, dc in dirs:
#             rr, cc = r+dr, c+dc
#             while 0 <= rr < 8 and 0 <= cc < 8:
#                 if board[rr][cc] is None:
#                     moves.append(Move((r,c),(rr,cc),board))
#                 else:
#                     if board[rr][cc].color != self.color:
#                         moves.append(Move((r,c),(rr,cc),board))
#                     break
#                 rr += dr; cc += dc
#         return moves

# class Bishop(Piece):
#     def __init__(self, color):
#         super().__init__(color)
#         self.name = 'B'
#     def get_valid_moves(self, r, c, board, en_passant_target, castle_rights):
#         moves = []
#         dirs = [(-1,-1),(-1,1),(1,-1),(1,1)]
#         for dr, dc in dirs:
#             rr, cc = r+dr, c+dc
#             while 0 <= rr < 8 and 0 <= cc < 8:
#                 if board[rr][cc] is None:
#                     moves.append(Move((r,c),(rr,cc),board))
#                 else:
#                     if board[rr][cc].color != self.color:
#                         moves.append(Move((r,c),(rr,cc),board))
#                     break
#                 rr += dr; cc += dc
#         return moves

# class Knight(Piece):
#     def __init__(self, color):
#         super().__init__(color)
#         self.name = 'N'
#     def get_valid_moves(self, r, c, board, en_passant_target, castle_rights):
#         moves = []
#         deltas = [(-2,-1),(-2,1),(-1,-2),(-1,2),(1,-2),(1,2),(2,-1),(2,1)]
#         for dr, dc in deltas:
#             rr, cc = r+dr, c+dc
#             if 0 <= rr < 8 and 0 <= cc < 8:
#                 if board[rr][cc] is None or board[rr][cc].color != self.color:
#                     moves.append(Move((r,c),(rr,cc),board))
#         return moves

# class Pawn(Piece):
#     def __init__(self, color):
#         super().__init__(color)
#         self.name = 'P'
#     def get_valid_moves(self, r, c, board, en_passant_target, castle_rights):
#         moves = []
#         direction = -1 if self.color == 'w' else 1
#         start_row = 6 if self.color == 'w' else 1
#         # Forward
#         if board[r+direction][c] is None:
#             moves.append(Move((r,c),(r+direction,c),board))
#             if r == start_row and board[r+2*direction][c] is None:
#                 moves.append(Move((r,c),(r+2*direction,c),board))
#         # Captures
#         for dc in (-1,1):
#             rr, cc = r+direction, c+dc
#             if 0 <= rr < 8 and 0 <= cc < 8:
#                 if board[rr][cc] and board[rr][cc].color != self.color:
#                     moves.append(Move((r,c),(rr,cc),board))
#                 elif (rr,cc) == en_passant_target:
#                     moves.append(Move((r,c),(rr,cc),board, is_en_passant=True))
#         return moves

# class Evaluation:
#     @staticmethod
#     def evaluate(board_state):
#         score = 0
#         for row in board_state:
#             for p in row:
#                 if p:
#                     val = PIECE_WEIGHTS[p.name]
#                     score += val if p.color=='w' else -val
#         return score

# class Player:
#     def __init__(self, color): self.color = color
#     def get_move(self, game): raise NotImplementedError

# class HumanPlayer(Player):
#     def get_move(self, game):
#         game.move_ready.clear()
#         game.move_ready.wait()
#         return game.user_move

# class AIPlayer(Player):
#     def __init__(self, color, depth=3):
#         super().__init__(color)
#         self.depth = depth
#     def get_move(self, game):
#         moves = game.board.get_all_valid_moves(self.color)
#         best_move, best_score = None, (float('-inf') if self.color=='w' else float('inf'))
#         alpha, beta = float('-inf'), float('inf')
#         for m in moves:
#             game.board.make_move(m, skip_update=True)
#             score = self.minimax(game.board, self.depth-1, alpha, beta, self.color!='w')
#             game.board.undo_move()
#             if (self.color=='w' and score > best_score) or (self.color=='b' and score < best_score):
#                 best_score, best_move = score, m
#             if self.color=='w': alpha = max(alpha, best_score)
#             else:         beta  = min(beta, best_score)
#         return best_move

#     def minimax(self, board_obj, depth, alpha, beta, maximizing):
#         if depth==0 or board_obj.checkmate or board_obj.stalemate:
#             return Evaluation.evaluate(board_obj.board)
#         color = 'w' if maximizing else 'b'
#         moves = board_obj.get_all_valid_moves(color)
#         if maximizing:
#             max_eval = float('-inf')
#             for m in moves:
#                 board_obj.make_move(m, skip_update=True)
#                 val = self.minimax(board_obj, depth-1, alpha, beta, False)
#                 board_obj.undo_move()
#                 max_eval = max(max_eval, val)
#                 alpha = max(alpha, val)
#                 if beta <= alpha: break
#             return max_eval
#         else:
#             min_eval = float('inf')
#             for m in moves:
#                 board_obj.make_move(m, skip_update=True)
#                 val = self.minimax(board_obj, depth-1, alpha, beta, True)
#                 board_obj.undo_move()
#                 min_eval = min(min_eval, val)
#                 beta = min(beta, val)
#                 if beta <= alpha: break
#             return min_eval

# class Board:
#     def __init__(self):
#         self.board = [[None]*8 for _ in range(8)]
#         self.white_to_move = True
#         self.move_log = []
#         self.en_passant_target = None
#         self.castle_rights = {'w':{'K':True,'Q':True}, 'b':{'K':True,'Q':True}}
#         self.checkmate = False
#         self.stalemate = False
#         self.setup_board()

#     def setup_board(self):
#         order = [Rook, Knight, Bishop, Queen, King, Bishop, Knight, Rook]
#         for c, cls in enumerate(order):
#             self.board[7][c] = cls('w')
#             self.board[0][c] = cls('b')
#         for c in range(8): self.board[6][c] = Pawn('w'); self.board[1][c] = Pawn('b')

#     def make_move(self, move, *, skip_update=False):
#         move.prev_en_passant = self.en_passant_target
#         move.prev_castle_rights = copy.deepcopy(self.castle_rights)
#         piece = self.board[move.start_row][move.start_col]
#         self.board[move.end_row][move.end_col] = piece
#         self.board[move.start_row][move.start_col] = None
#         # Pawn promotion
#         if isinstance(piece, Pawn) and (move.end_row in (0,7)):
#             self.board[move.end_row][move.end_col] = Queen(piece.color)
#         # En passant
#         if move.is_en_passant:
#             flop = 1 if piece.color=='w' else -1
#             self.board[move.end_row+flop][move.end_col] = None
#         # Castling
#         if move.is_castle:
#             if move.end_col - move.start_col == 2:
#                 self.board[move.end_row][5] = self.board[move.end_row][7]
#                 self.board[move.end_row][7] = None
#             else:
#                 self.board[move.end_row][3] = self.board[move.end_row][0]
#                 self.board[move.end_row][0] = None
#         # Update en passant target
#         if isinstance(piece, Pawn) and abs(move.end_row - move.start_row) == 2:
#             self.en_passant_target = ((move.start_row + move.end_row)//2, move.start_col)
#         else:
#             self.en_passant_target = None
#         # Update castling rights
#         self.update_castle_rights(move)
#         self.move_log.append(move)
#         self.white_to_move = not self.white_to_move
#         if not skip_update:
#             self.update_game_end()

#     def undo_move(self):
#         if not self.move_log: return
#         move = self.move_log.pop()
#         self.board[move.start_row][move.start_col] = move.piece_moved
#         self.board[move.end_row][move.end_col] = move.piece_captured
#         self.en_passant_target = move.prev_en_passant
#         self.castle_rights = move.prev_castle_rights
#         self.white_to_move = not self.white_to_move
#         self.checkmate = False; self.stalemate = False

#     def update_castle_rights(self, move):
#         piece = move.piece_moved
#         if isinstance(piece, King): self.castle_rights[piece.color] = {'K':False,'Q':False}
#         elif isinstance(piece, Rook):
#             if move.start_col == 0: self.castle_rights[piece.color]['Q'] = False
#             if move.start_col == 7: self.castle_rights[piece.color]['K'] = False

#     def get_all_valid_moves(self, color):
#         moves = []
#         for r in range(8):
#             for c in range(8):
#                 p = self.board[r][c]
#                 if p and p.color == color:
#                     for m in p.get_valid_moves(r,c,self.board,self.en_passant_target,self.castle_rights):
#                         self.make_move(m, skip_update=True)
#                         if not self.is_in_check(color): moves.append(m)
#                         self.undo_move()
#         return moves

#     def is_in_check(self, color):
#         # find king
#         for r in range(8):
#             for c in range(8):
#                 p = self.board[r][c]
#                 if isinstance(p, King) and p.color == color:
#                     return self.square_attacked(r,c,'b' if color=='w' else 'w')
#         return False

#     def square_attacked(self, r, c, enemy_color):
#         for rr in range(8):
#             for cc in range(8):
#                 p = self.board[rr][cc]
#                 if p and p.color == enemy_color:
#                     for m in p.get_valid_moves(rr,cc,self.board,self.en_passant_target,self.castle_rights):
#                         if m.end_row==r and m.end_col==c: return True
#         return False

#     def update_game_end(self):
#         color = 'w' if self.white_to_move else 'b'
#         moves = self.get_all_valid_moves(color)
#         if not moves:
#             if self.is_in_check(color): self.checkmate = True
#             else: self.stalemate = True

# class ChessGame:
#     def __init__(self):
#         self.board = Board()
#         self.root = tk.Tk()
#         self.root.title("Chess vs AI")
#         self.canvas = tk.Canvas(self.root, width=520, height=520)
#         self.canvas.pack()
#         self.status = tk.Label(self.root, text="White to move")
#         self.status.pack()
#         self.entry = tk.Entry(self.root)
#         self.entry.pack()
#         self.button = tk.Button(self.root, text="Make Move", command=self.on_enter)
#         self.button.pack()
#         self.move_ready = threading.Event()
#         self.user_move = None
#         self.players = {'w': HumanPlayer('w'), 'b': AIPlayer('b', depth=3)}
#         self.load_images()
#         self.draw_board()
#         threading.Thread(target=self.game_loop, daemon=True).start()
#         self.root.mainloop()

#     def load_images(self):
#         self.images = {
#             'wK':'\u2654','wQ':'\u2655','wR':'\u2656','wB':'\u2657','wN':'\u2658','wP':'\u2659',
#             'bK':'\u265A','bQ':'\u265B','bR':'\u265C','bB':'\u265D','bN':'\u265E','bP':'\u265F'
#         }

#     def draw_board(self):
#         self.canvas.delete("all")
#         colors = ["#EEEED2","#769656"]
#         for r in range(8):
#             for c in range(8):
#                 x1,y1 = c*65, r*65; x2,y2 = x1+65, y1+65
#                 self.canvas.create_rectangle(x1,y1,x2,y2, fill=colors[(r+c)%2])
#                 p = self.board.board[r][c]
#                 if p:
#                     symbol = self.images[p.color + p.name]
#                     fill = "white" if p.color=='w' else "black"
#                     self.canvas.create_text(x1+32,y1+32, text=symbol, font=("Arial",32), fill=fill)

#     def on_enter(self):
#         mv = self.entry.get().strip().lower().replace(" ","")
#         if len(mv)==4 and mv[0] in FILES and mv[2] in FILES and mv[1] in RANKS and mv[3] in RANKS:
#             start = (7-int(mv[1]), FILES.index(mv[0]))
#             end   = (7-int(mv[3]), FILES.index(mv[2]))
#             self.user_move = Move(start, end, self.board.board)
#             self.move_ready.set()
#             self.status.config(text="")
#         else:
#             self.status.config(text="Invalid move (e.g. e2e4)")
#         self.entry.delete(0, tk.END)

#     def game_loop(self):
#         while True:
#             color = 'w' if self.board.white_to_move else 'b'
#             player = self.players[color]
#             if isinstance(player, AIPlayer):
#                 self.status.config(text="AI is thinking...")
#             mv = player.get_move(self)
#             self.board.make_move(mv)
#             self.draw_board()
#             if self.board.checkmate:
#                 winner = 'White' if color=='b' else 'Black'
#                 self.status.config(text=f"Checkmate! {winner} wins.")
#                 break
#             if self.board.stalemate:
#                 self.status.config(text="Stalemate!")
#                 break
#             nxt = 'White' if not self.board.white_to_move else 'Black'
#             self.status.config(text=f"{nxt} to move")
#             time.sleep(0.1)

# if __name__=='__main__':
#     ChessGame()





import tkinter as tk
import threading
import copy
import time

# Constants
PIECE_WEIGHTS = {'K': 0, 'Q': 9, 'R': 5, 'B': 3, 'N': 3, 'P': 1}
FILES = "abcdefgh"
RANKS = "12345678"
SQUARE_SIZE = 65
BOARD_COLOR_LIGHT = "#EEEED2"
BOARD_COLOR_DARK = "#769656"
HIGHLIGHT_COLOR = "#BACA44"

class Move:
    def __init__(self, start_sq, end_sq, board, is_en_passant=False, is_castle=False, promotion_choice=None):
        self.start_row, self.start_col = start_sq
        self.end_row, self.end_col = end_sq
        self.piece_moved = board[self.start_row][self.start_col]
        self.piece_captured = board[self.end_row][self.end_col]
        self.is_en_passant = is_en_passant
        self.is_castle = is_castle
        self.promotion_choice = promotion_choice
        self.prev_en_passant = None
        self.prev_castle_rights = None

    def get_chess_notation(self):
        return f"{FILES[self.start_col]}{RANKS[7-self.start_row]}{FILES[self.end_col]}{RANKS[7-self.end_row]}" + (
            f"={self.promotion_choice}" if self.promotion_choice else "")

class Piece:
    def __init__(self, color):
        self.color = color
    def get_valid_moves(self, row, col, board, en_passant_target, castle_rights):
        raise NotImplementedError

class King(Piece):
    name = 'K'
    def get_valid_moves(self, r, c, board, en_passant_target, castle_rights):
        moves = []
        directions = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
        for dr, dc in directions:
            rr, cc = r+dr, c+dc
            if 0 <= rr < 8 and 0 <= cc < 8:
                target = board[rr][cc]
                if target is None or target.color != self.color:
                    moves.append(Move((r,c),(rr,cc),board))
        rights = castle_rights[self.color]
        # Kingside
        if rights['K'] and board[r][c+1] is None and board[r][c+2] is None:
            moves.append(Move((r,c),(r,c+2),board,is_castle=True))
        # Queenside
        if rights['Q'] and board[r][c-1] is None and board[r][c-2] is None and board[r][c-3] is None:
            moves.append(Move((r,c),(r,c-2),board,is_castle=True))
        return moves

class Queen(Piece):
    name = 'Q'
    def get_valid_moves(self, r, c, board, en_passant_target, castle_rights):
        return Rook(self.color).get_valid_moves(r,c,board,en_passant_target,castle_rights) + \
               Bishop(self.color).get_valid_moves(r,c,board,en_passant_target,castle_rights)

class Rook(Piece):
    name = 'R'
    def get_valid_moves(self, r, c, board, en_passant_target, castle_rights):
        moves = []
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            rr, cc = r+dr, c+dc
            while 0 <= rr < 8 and 0 <= cc < 8:
                if board[rr][cc] is None:
                    moves.append(Move((r,c),(rr,cc),board))
                else:
                    if board[rr][cc].color != self.color:
                        moves.append(Move((r,c),(rr,cc),board))
                    break
                rr += dr; cc += dc
        return moves

class Bishop(Piece):
    name = 'B'
    def get_valid_moves(self, r, c, board, en_passant_target, castle_rights):
        moves = []
        for dr, dc in [(-1,-1),(-1,1),(1,-1),(1,1)]:
            rr, cc = r+dr, c+dc
            while 0 <= rr < 8 and 0 <= cc < 8:
                if board[rr][cc] is None:
                    moves.append(Move((r,c),(rr,cc),board))
                else:
                    if board[rr][cc].color != self.color:
                        moves.append(Move((r,c),(rr,cc),board))
                    break
                rr += dr; cc += dc
        return moves

class Knight(Piece):
    name = 'N'
    def get_valid_moves(self, r, c, board, en_passant_target, castle_rights):
        moves = []
        for dr, dc in [(-2,-1),(-2,1),(-1,-2),(-1,2),(1,-2),(1,2),(2,-1),(2,1)]:
            rr, cc = r+dr, c+dc
            if 0 <= rr < 8 and 0 <= cc < 8:
                if board[rr][cc] is None or board[rr][cc].color != self.color:
                    moves.append(Move((r,c),(rr,cc),board))
        return moves

class Pawn(Piece):
    name = 'P'
    def get_valid_moves(self, r, c, board, en_passant_target, castle_rights):
        moves = []
        dir = -1 if self.color=='w' else 1
        start = 6 if self.color=='w' else 1
        if board[r+dir][c] is None:
            moves.append(Move((r,c),(r+dir,c),board))
            if r==start and board[r+2*dir][c] is None:
                moves.append(Move((r,c),(r+2*dir,c),board))
        for dc in (-1,1):
            rr, cc = r+dir, c+dc
            if 0 <= rr < 8 and 0 <= cc < 8:
                if board[rr][cc] and board[rr][cc].color!=self.color:
                    moves.append(Move((r,c),(rr,cc),board))
                elif (rr,cc)==en_passant_target:
                    moves.append(Move((r,c),(rr,cc),board,is_en_passant=True))
        return moves

class Evaluation:
    @staticmethod
    def evaluate(board_state):
        score = 0
        for row in board_state:
            for p in row:
                if p:
                    val = PIECE_WEIGHTS[p.name]
                    score += val if p.color=='w' else -val
        return score

class Player:
    def __init__(self, color): self.color=color
    def get_move(self, game): raise NotImplementedError

class HumanPlayer(Player):
    def get_move(self, game):
        game.move_ready.clear()
        game.move_ready.wait()
        return game.user_move

class AIPlayer(Player):
    def __init__(self, color, depth=3):
        super().__init__(color)
        self.depth=depth
    def get_move(self, game):
        moves = game.board.get_all_valid_moves(self.color)
        best_move, best_score = None, (float('-inf') if self.color=='w' else float('inf'))
        alpha, beta = float('-inf'), float('inf')
        for m in moves:
            game.board.make_move(m, skip_update=True)
            score = self.minimax(game.board, self.depth-1, alpha, beta, self.color!='w')
            game.board.undo_move()
            if (self.color=='w' and score>best_score) or (self.color=='b' and score<best_score):
                best_score, best_move = score, m
            if self.color=='w': alpha=max(alpha,best_score)
            else: beta=min(beta,best_score)
        return best_move
    def minimax(self, board, depth, alpha, beta, maximizing):
        if depth==0 or board.checkmate or board.stalemate:
            return Evaluation.evaluate(board.board)
        color='w' if maximizing else 'b'
        moves = board.get_all_valid_moves(color)
        if maximizing:
            max_eval=float('-inf')
            for m in moves:
                board.make_move(m, skip_update=True)
                val=self.minimax(board, depth-1, alpha, beta, False)
                board.undo_move()
                max_eval=max(max_eval,val)
                alpha=max(alpha,val)
                if beta<=alpha: break
            return max_eval
        else:
            min_eval=float('inf')
            for m in moves:
                board.make_move(m, skip_update=True)
                val=self.minimax(board, depth-1, alpha, beta, True)
                board.undo_move()
                min_eval=min(min_eval,val)
                beta=min(beta,val)
                if beta<=alpha: break
            return min_eval

class Board:
    def __init__(self):
        self.board=[[None]*8 for _ in range(8)]
        self.white_to_move=True
        self.move_log=[]
        self.en_passant_target=None
        self.castle_rights={'w':{'K':True,'Q':True},'b':{'K':True,'Q':True}}
        self.checkmate=False
        self.stalemate=False
        self.setup_board()
    def setup_board(self):
        order=[Rook,Knight,Bishop,Queen,King,Bishop,Knight,Rook]
        for c,cls in enumerate(order): self.board[7][c]=cls('w'); self.board[0][c]=cls('b')
        for c in range(8): self.board[6][c]=Pawn('w'); self.board[1][c]=Pawn('b')
    def make_move(self, move, *, skip_update=False):
        move.prev_en_passant=self.en_passant_target
        move.prev_castle_rights=copy.deepcopy(self.castle_rights)
        piece=self.board[move.start_row][move.start_col]
        self.board[move.end_row][move.end_col]=piece
        self.board[move.start_row][move.start_col]=None
        if isinstance(piece,Pawn) and move.end_row in (0,7): self.board[move.end_row][move.end_col]=Queen(piece.color)
        if move.is_en_passant:
            dir=1 if piece.color=='w' else -1
            self.board[move.end_row+dir][move.end_col]=None
        if move.is_castle:
            if move.end_col-move.start_col==2:
                self.board[move.end_row][5]=self.board[move.end_row][7]; self.board[move.end_row][7]=None
            else:
                self.board[move.end_row][3]=self.board[move.end_row][0]; self.board[move.end_row][0]=None
        if isinstance(piece,Pawn) and abs(move.end_row-move.start_row)==2:
            self.en_passant_target=((move.start_row+move.end_row)//2,move.start_col)
        else: self.en_passant_target=None
        self.update_castle_rights(move)
        self.move_log.append(move)
        self.white_to_move=not self.white_to_move
        if not skip_update: self.update_game_end()
    def undo_move(self):
        if not self.move_log: return
        move=self.move_log.pop()
        self.board[move.start_row][move.start_col]=move.piece_moved
        self.board[move.end_row][move.end_col]=move.piece_captured
        self.en_passant_target=move.prev_en_passant
        self.castle_rights=move.prev_castle_rights
        self.white_to_move=not self.white_to_move
        self.checkmate=self.stalemate=False
    def update_castle_rights(self, move):
        piece=move.piece_moved
        if isinstance(piece,King): self.castle_rights[piece.color]={'K':False,'Q':False}
        elif isinstance(piece,Rook):
            if move.start_col==0: self.castle_rights[piece.color]['Q']=False
            if move.start_col==7: self.castle_rights[piece.color]['K']=False
    def get_all_valid_moves(self, color):
        moves=[]
        for r in range(8):
            for c in range(8):
                p=self.board[r][c]
                if p and p.color==color:
                    for m in p.get_valid_moves(r,c,self.board,self.en_passant_target,self.castle_rights):
                        self.make_move(m, skip_update=True)
                        if not self.is_in_check(color): moves.append(m)
                        self.undo_move()
        return moves
    def is_in_check(self,color):
        for r in range(8):
            for c in range(8):
                p=self.board[r][c]
                if isinstance(p,King) and p.color==color:
                    return self.square_attacked(r,c,'b' if color=='w' else 'w')
        return False
    def square_attacked(self,r,c,enemy_color):
        for rr in range(8):
            for cc in range(8):
                p=self.board[rr][cc]
                if p and p.color==enemy_color:
                    for m in p.get_valid_moves(rr,cc,self.board,self.en_passant_target,self.castle_rights):
                        if m.end_row==r and m.end_col==c: return True
        return False
    def update_game_end(self):
        color='w' if self.white_to_move else 'b'
        moves=self.get_all_valid_moves(color)
        if not moves:
            if self.is_in_check(color): self.checkmate=True
            else: self.stalemate=True

class ChessGame:
    def __init__(self):
        self._init_ui()
        self._start_game_loop()
        self.root.mainloop()
        
    def _init_ui(self):
        self.board = Board()
        self.root = tk.Tk()
        self.root.title("Chess Game")
        self.root.configure(bg="#f0f0f0")
        
        # Create menu
        menubar = tk.Menu(self.root)
        game_menu = tk.Menu(menubar, tearoff=0)
        game_menu.add_command(label="New Game", command=self.reset_game)
        game_menu.add_separator()
        game_menu.add_command(label="Exit", command=self.root.destroy)
        menubar.add_cascade(label="Game", menu=game_menu)
        self.root.config(menu=menubar)
        
        # Main frame
        main_frame = tk.Frame(self.root, bg="#f0f0f0", padx=10, pady=10)
        main_frame.pack(expand=True, fill=tk.BOTH)
        
        # Left panel for board
        left_panel = tk.Frame(main_frame, bg="#f0f0f0")
        left_panel.pack(side=tk.LEFT, padx=10)
        
        # Board canvas
        self.canvas = tk.Canvas(left_panel, width=8*SQUARE_SIZE, height=8*SQUARE_SIZE)
        self.canvas.pack()
        
        # Right panel for controls and info
        right_panel = tk.Frame(main_frame, bg="#f0f0f0", padx=10, pady=10)
        right_panel.pack(side=tk.RIGHT, fill=tk.Y)
        
        # Status display
        status_frame = tk.LabelFrame(right_panel, text="Game Status", bg="#f0f0f0", padx=5, pady=5)
        status_frame.pack(fill=tk.X, pady=10)
        
        self.status = tk.Label(status_frame, text="White to move", font=("Arial", 12), bg="#f0f0f0")
        self.status.pack(pady=5)
        
        # Move history
        history_frame = tk.LabelFrame(right_panel, text="Move History", bg="#f0f0f0", padx=5, pady=5)
        history_frame.pack(fill=tk.X, pady=10)
        
        self.history_text = tk.Text(history_frame, width=20, height=10, font=("Courier", 10))
        self.history_text.pack(fill=tk.BOTH, expand=True)
        
        # Manual move entry
        move_frame = tk.LabelFrame(right_panel, text="Enter Move", bg="#f0f0f0", padx=5, pady=5)
        move_frame.pack(fill=tk.X, pady=10)
        
        self.entry = tk.Entry(move_frame, font=("Arial", 12))
        self.entry.pack(fill=tk.X, pady=5)
        self.entry.bind("<Return>", lambda e: self.on_enter())
        
        button_frame = tk.Frame(move_frame, bg="#f0f0f0")
        button_frame.pack(fill=tk.X)
        
        self.button = tk.Button(button_frame, text="Make Move", 
                               command=self.on_enter, 
                               bg="#4CAF50", fg="white",
                               font=("Arial", 10, "bold"),
                               padx=10)
        self.button.pack(side=tk.LEFT, expand=True, fill=tk.X, pady=5)
        
        undo_button = tk.Button(button_frame, text="Undo",
                              command=self.undo_move,
                              bg="#FF9800", fg="white",
                              font=("Arial", 10, "bold"),
                              padx=10)
        undo_button.pack(side=tk.RIGHT, expand=True, fill=tk.X, pady=5)
        
        # Bind canvas events
        self.canvas.bind('<Button-1>', self.on_canvas_click)
        
        # Game variables
        self.move_ready = threading.Event()
        self.user_move = None
        self.players = {'w': HumanPlayer('w'), 'b': AIPlayer('b', depth=3)}
        self.selected_sq = None
        self.valid_moves = []
        self.last_move = None
        
        # Load piece images
        self.load_images()
        self.draw_board()
        
    def _start_game_loop(self):
        threading.Thread(target=self.game_loop, daemon=True).start()
    
    def reset_game(self):
        self.board = Board()
        self.selected_sq = None
        self.valid_moves = []
        self.last_move = None
        self.history_text.delete(1.0, tk.END)
        self.draw_board()
        threading.Thread(target=self.game_loop, daemon=True).start()
    
    def undo_move(self):
        if self.board.move_log:
            self.board.undo_move()
            if self.board.move_log:
                self.last_move = self.board.move_log[-1]
            else:
                self.last_move = None
            self.draw_board()
            if len(self.history_text.get(1.0, tk.END).strip().split("\n")) > 1:
                lines = self.history_text.get(1.0, tk.END).strip().split("\n")
                self.history_text.delete(1.0, tk.END)
                self.history_text.insert(tk.END, "\n".join(lines[:-1]))
            color = 'White' if self.board.white_to_move else 'Black'
            self.status.config(text=f"{color}'s turn")
    
    def load_images(self):
        self.images = {
            'wK':'\u2654', 'wQ':'\u2655', 'wR':'\u2656', 'wB':'\u2657', 'wN':'\u2658', 'wP':'\u2659',
            'bK':'\u265A', 'bQ':'\u265B', 'bR':'\u265C', 'bB':'\u265D', 'bN':'\u265E', 'bP':'\u265F'
        }
    
    def draw_board(self):
        self.canvas.delete("all")
        
        # Draw board squares
        for r in range(8):
            for c in range(8):
                x1, y1 = c * SQUARE_SIZE, r * SQUARE_SIZE
                x2, y2 = x1 + SQUARE_SIZE, y1 + SQUARE_SIZE
                color = BOARD_COLOR_LIGHT if (r + c) % 2 == 0 else BOARD_COLOR_DARK
                self.canvas.create_rectangle(x1, y1, x2, y2, fill=color, outline="")
                
                # Draw rank and file labels
                if c == 0:  # File labels (left side)
                    text_color = BOARD_COLOR_DARK if r % 2 == 0 else BOARD_COLOR_LIGHT
                    self.canvas.create_text(5, y1 + SQUARE_SIZE/2, 
                                          text=RANKS[7-r], 
                                          fill=text_color, 
                                          anchor="w",
                                          font=("Arial", 10))
                if r == 7:  # Rank labels (bottom)
                    text_color = BOARD_COLOR_DARK if c % 2 == 0 else BOARD_COLOR_LIGHT
                    self.canvas.create_text(x1 + SQUARE_SIZE/2, y2 - 5, 
                                          text=FILES[c], 
                                          fill=text_color,
                                          anchor="s",
                                          font=("Arial", 10))
        
        # Highlight last move
        if self.last_move:
            start_col, start_row = self.last_move.start_col, self.last_move.start_row
            end_col, end_row = self.last_move.end_col, self.last_move.end_row
            
            # Highlight start square with a subtle orange
            x1, y1 = start_col * SQUARE_SIZE, start_row * SQUARE_SIZE
            self.canvas.create_rectangle(x1, y1, x1 + SQUARE_SIZE, y1 + SQUARE_SIZE,
                                       fill="#F4C430", stipple="gray50")
            
            # Highlight end square with a bolder orange
            x1, y1 = end_col * SQUARE_SIZE, end_row * SQUARE_SIZE
            self.canvas.create_rectangle(x1, y1, x1 + SQUARE_SIZE, y1 + SQUARE_SIZE,
                                       fill="#F4C430", stipple="gray25")
        
        # Highlight selected square
        if self.selected_sq:
            sr, sc = self.selected_sq
            x1, y1 = sc * SQUARE_SIZE, sr * SQUARE_SIZE
            self.canvas.create_rectangle(x1, y1, x1 + SQUARE_SIZE, y1 + SQUARE_SIZE,
                                       outline=HIGHLIGHT_COLOR, width=3)
            
            # Highlight valid moves
            for m in self.valid_moves:
                if m.start_row == sr and m.start_col == sc:
                    x, y = m.end_col * SQUARE_SIZE, m.end_row * SQUARE_SIZE
                    if self.board.board[m.end_row][m.end_col]:  # Capture
                        # Draw a circle outline for captures
                        self.canvas.create_oval(x + 5, y + 5, x + SQUARE_SIZE - 5, y + SQUARE_SIZE - 5,
                                             outline=HIGHLIGHT_COLOR, width=3)
                    else:  # Regular move
                        # Draw a dot for empty square moves
                        self.canvas.create_oval(x + SQUARE_SIZE/2 - 8, y + SQUARE_SIZE/2 - 8,
                                             x + SQUARE_SIZE/2 + 8, y + SQUARE_SIZE/2 + 8,
                                             fill=HIGHLIGHT_COLOR)
        
        # Draw pieces
        for r in range(8):
            for c in range(8):
                p = self.board.board[r][c]
                if p:
                    sym = self.images[p.color + p.name]
                    col = 'white' if p.color == 'w' else 'black'
                    self.canvas.create_text(c * SQUARE_SIZE + SQUARE_SIZE/2,
                                          r * SQUARE_SIZE + SQUARE_SIZE/2,
                                          text=sym, font=("Arial", 32), fill=col)
    
    def on_canvas_click(self, event):
        col, row = event.x // SQUARE_SIZE, event.y // SQUARE_SIZE
        if not (0 <= col < 8 and 0 <= row < 8):
            return
            
        color = 'w' if self.board.white_to_move else 'b'
        
        # If no piece is selected yet
        if not self.selected_sq:
            p = self.board.board[row][col]
            if p and p.color == color:
                self.selected_sq = (row, col)
                all_moves = self.board.get_all_valid_moves(color)
                self.valid_moves = [m for m in all_moves if (m.start_row, m.start_col) == self.selected_sq]
                self.draw_board()
        else:  # If a piece is already selected
            # Check if clicked on a valid destination square
            mv = next((m for m in self.valid_moves 
                      if m.end_row == row and m.end_col == col), None)
            
            if mv:
                # If valid move, make it
                self.user_move = mv
                self.move_ready.set()
            elif self.board.board[row][col] and self.board.board[row][col].color == color:
                # If clicked on another piece of same color, select that piece instead
                self.selected_sq = (row, col)
                all_moves = self.board.get_all_valid_moves(color)
                self.valid_moves = [m for m in all_moves if (m.start_row, m.start_col) == self.selected_sq]
            else:
                # If clicked elsewhere, deselect
                self.selected_sq = None
                self.valid_moves = []
            
            self.draw_board()
    
    def on_enter(self):
        mv = self.entry.get().strip().lower().replace(" ", "")
        if len(mv) == 4 and mv[0] in FILES and mv[2] in FILES and mv[1] in RANKS and mv[3] in RANKS:
            start = (7 - int(mv[1]), FILES.index(mv[0]))
            end = (7 - int(mv[3]), FILES.index(mv[2]))
            self.user_move = Move(start, end, self.board.board)
            self.move_ready.set()
            self.status.config(text="")
        else:
            self.status.config(text="Invalid move (e.g. e2e4)")
        self.entry.delete(0, tk.END)
    
    def game_loop(self):
        move_num = 1
        
        while True:
            color = 'w' if self.board.white_to_move else 'b'
            player = self.players[color]
            
            if isinstance(player, AIPlayer):
                self.status.config(text="AI is thinking...")
                self.root.update()  # Force GUI update
            
            m = player.get_move(self)
            if m:
                self.board.make_move(m)
                self.last_move = m
                
                # Add move to history
                move_text = m.get_chess_notation()
                if color == 'w':
                    self.history_text.insert(tk.END, f"{move_num}. {move_text} ")
                else:
                    self.history_text.insert(tk.END, f"{move_text}\n")
                    move_num += 1
                self.history_text.see(tk.END)  # Scroll to latest move
            
            self.selected_sq = None
            self.valid_moves = []
            self.draw_board()
            
            # Check game end conditions
            if self.board.checkmate:
                winner = 'White' if color == 'b' else 'Black'
                self.status.config(text=f"Checkmate! {winner} wins.")
                break
                
            if self.board.stalemate:
                self.status.config(text="Stalemate! Game drawn.")
                break
                
            # If game continues, update status for next player
            next_color = 'White' if not self.board.white_to_move else 'Black'
            self.status.config(text=f"{next_color}'s turn")
            
            time.sleep(0.1)

if __name__=='__main__':
    ChessGame()




